# 1. Generative AI Model Selection & Setup

### 1.1 Why Generative AI?

The supervised learning model built in Phase 1 (Random Forest) outputs a fit label — **Fit**, **Small**, or **Large**. While this is useful, a raw label is not particularly helpful to a real shopper. The goal of integrating Generative AI is to translate that prediction into a natural language explanation that is personalized, readable, and actionable.

For example, instead of showing a user `"Small"`, the system should be able to say:
> *"Based on your measurements, this item tends to run small for your body type. You may want to consider sizing up to a medium."*

This makes the system far more usable for non-technical end users.

### 1.2 Model Candidates Considered

We evaluated three potential models before selecting one:

| Model | Provider | Access | Strengths | Weaknesses |
|---|---|---|---|---|
| GPT-4o-mini | OpenAI | Paid API | Strong instruction following, reliable outputs, well-documented | Costs money per token, requires OpenAI account |
| LLaMA 3 (8B) | Meta (via Groq) | Free API | Open-source, fast via Groq, no per-token cost | Less consistent on varying prompts, setup more involved |
| Gemini 1.5 Flash | Google | Free tier API | Good for short structured outputs, generous free tier | Free tier data may be used for improvement |

### 1.3 Selected Model: Gemini 1.5 Flash (Google)

We selected Gemini 1.5 Flash as our primary model for the following reasons:

- **Zero Cost**: Gemini offers a robust free tier via Google AI Studio that allows for 1,500 requests per day without any financial commitment.
- **Instruction Following**: It handles structured prompt templates and scaled numerical features with high precision, ensuring the _scaled inputs are interpreted correctly.
- **Generous Rate Limits**: The 15 requests-per-minute (RPM) allowance on the free tier is more than sufficient for our development and testing phases at university.
- **Ease of Integration**: The Google Generative AI Python SDK is straightforward to implement, and the API key setup does not require a credit card for the free tier.

LLaMA 3.3 via Groq remains a viable fallback option if ultra-low latency becomes the priority, though Gemini is preferred for its stability and higher daily request quota.

### 1.4 API Setup

The API key is stored in a `.env` file and loaded using `python-dotenv`. The key is **never hardcoded** in this notebook. See `api_config_template.py` for the setup template.

Required libraries:
```
openai
python-dotenv
```

In [1]:
# pip install google-generativeai python-dotenv

import os
import json
from dotenv import load_dotenv
import google.generativeai as genai

load_dotenv()
genai.configure(api_key=os.environ["GEMINI_API_KEY"])
model = genai.GenerativeModel("gemini-1.5-flash")

print("Gemini ready:", "GEMINI_API_KEY" in os.environ)

Gemini ready: True


c:\Users\kadij\OneDrive\Desktop\AI project\SWE485-Project-Group10\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\kadij\AppData\Local\Temp\ipykernel_16936\2910145782.py:6: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


# 2. Prompt Template Design Documentation

We designed four prompt templates, each with a distinct strategy. The templates progressively incorporate more context, from a minimal baseline to a full measurement-and-cluster-informed guide. This design allows us to compare how much context the model actually needs to produce useful output.

All templates are saved as `.json` files in `/Generative_AI/prompts/`.

---
### Template 1 – Basic Prediction Explainer

**Template ID:** T1  
**Intended Use Case:** Minimal-context explanation of the fit label. This template assumes we have no user data available — only the raw prediction from the ML model.  

**Design Rationale:**  
This serves as our baseline. Before adding any user data, we want to understand how well the model can explain a fit prediction on its own. Many real scenarios involve incomplete user profiles, so a baseline that works with just a label is practically valuable. It also helps isolate the effect of adding context in later templates.

**Prompt Structure:**
```
A clothing size recommendation system predicted that a '{prediction}' fit label applies
to this purchase. In 2-3 sentences, explain what this means for the customer in simple,
friendly language. Do not suggest any action, just explain the prediction.
```

**Placeholders:**
- `{prediction}`: fit label from ML model: `Fit`, `Small`, or `Large`

**Example Input:**
```json
{ "prediction": "Small" }
```

**Example Output (expected):**
> *"The system predicted that this item runs small, meaning the size you selected may feel tighter or shorter than expected. This is common with certain brands whose sizing doesn't align with standard measurements. It's worth keeping this in mind when finalizing your choice."*

**Assumptions & Limitations:**  
No personalization — the output is the same for every user with the same label. It is informative but generic. Useful as a fallback when user data is unavailable.

---
### Template 2 – Measurement-Aware Personalized Advice

**Template ID:** T2  
**Intended Use Case:** Personalized advice that incorporates the customer's actual body measurements alongside the prediction.  

**Design Rationale:**  
The dataset contains several body-related measurements — height, bra size, cup size, and hip measurements — along with the clothing length and size ordered. These are the exact features our Random Forest model used to make the prediction, so passing them back to the language model gives it the context to explain *why* the prediction was made, not just what it is. This is the most natural upgrade from T1 and produces advice that feels genuinely tailored to the individual.

**Prompt Structure:**
```
A customer with the following measurements is shopping online:
- Height: {height} cm
- Hips: {hips} inches
- Bra size: {bra_size}
- Cup size: {cup_size}
- Preferred clothing length: {length}
- Size ordered: {size}

Our ML model predicted the fit as '{prediction}' for the item they selected.
Based on these measurements and the prediction, give the customer a short, friendly
sizing recommendation (2-3 sentences). Be specific about their measurements.
```

**Placeholders:**
- `{height}` — customer height in cm (from `height_scaled`, reversed)
- `{hips}` — hip measurement in inches (from `hips_scaled`, reversed)
- `{bra_size}` — bra size e.g. `34`, `36` (from `bra size_scaled`, reversed)
- `{cup_size}` — cup size e.g. `B`, `C`, `D` (from `cup size_scaled`, reversed)
- `{length}` — preferred clothing length e.g. `just right`, `slightly long`, `very short` (from `length_scaled`, reversed)
- `{size}` — size ordered e.g. `7`, `24`, `25`, `33` (from `size_scaled`, reversed)
- `{prediction}` — fit label from ML model: `Fit`, `Small`, or `Large`

**Example Input:**
```json
{
  "height": 165,
  "hips": 40,
  "bra_size": 36,
  "cup_size": "C",
  "length": "just right",
  "size": "24",
  "prediction": "Large"
}
```

**Example Output (expected):**
> *"Based on your measurements, the medium you ordered is predicted to run large — meaning it will likely feel loose around your hips and bust area. For a 36C with 40-inch hips at your height, sizing down to a small would probably give you a better fit. This is especially common in regular-length tops and dresses where the waist cut tends to be generous."*

**Assumptions & Limitations:**  
Requires that scaled features are inverse-transformed before being passed to the prompt, so the model receives interpretable values rather than normalized numbers. If any measurement fields are missing in a user's record, the prompt will degrade, a fallback to T1 should be used in that case.

---
### Template 3 – Cluster-Informed Contextual Advice

**Template ID:** T3  
**Intended Use Case:** Advice informed by the customer's cluster group identified during unsupervised learning, where clusters were derived from the same body measurement features used in the classifier.  

**Design Rationale:**  
The clustering step groups customers with similar combinations of height, hip measurements, bra/cup size, and clothing length preference. These groups capture fit patterns that individual measurements alone may not express clearly — for example, a cluster of customers who are tall with larger hip measurements and consistently report items running small in the hips. Passing the cluster's characteristic profile to the language model allows it to generate advice grounded in group-level patterns rather than just interpreting one individual's numbers in isolation.

**Prompt Structure:**
```
Based on body measurements and clothing preferences, this customer belongs to a
customer group characterized by: {cluster_description}.

Their selected item received a predicted fit of '{prediction}'.

Using this group profile, write a 2-3 sentence sizing suggestion that reflects
what customers with these characteristics commonly experience when shopping for
clothing online.
```

**Placeholders:**
- `{cluster_description}` - a human-readable summary of the cluster's defining features. Should reference the actual measurement features e.g. *"above-average height (170+ cm), larger hip measurements (42+ inches), and a preference for regular-length items"*
- `{prediction}` - fit label from ML model: `Fit`, `Small`, or `Large`

**Example Input:**
```json
{
  "cluster_description": "above-average height (170+ cm), larger hip measurements (42+ inches), bra size 36–38, and a preference for regular-length clothing",
  "prediction": "Small"
}
```

**Example Output (expected):**
> *"Customers with your body profile — taller builds with fuller hips — often find that items run small through the hips and waist even when length fits well. Since this item was predicted to run small for you, sizing up by one would likely give a more comfortable fit in those areas. This is a pattern we commonly see for shoppers with a similar measurement profile."*

**Assumptions & Limitations:**  
The cluster description is written manually based on the centroid analysis done in Part A. Its quality depends directly on how well-separated and interpretable the clusters are. If clusters overlap significantly in feature space, the descriptions will be vague and the advice will not be meaningfully different from T2.

---
### Template 4 – Actionable Shopping Guide

**Template ID:** T4  
**Intended Use Case:** Full-context, actionable advice combining the customer's measurements, clothing category, quality rating, and the fit prediction into a practical shopping guide.  

**Design Rationale:**  
Templates 1–3 are explanatory. This template shifts toward telling the user what to *do*. It adds two fields that the others lack: `category` (the type of clothing item) and `quality` (the quality rating the customer associated with the item). Category matters because fit issues differ significantly across item types — a "Small" prediction in dresses has different implications than in tops or bottoms. Quality adds useful context because customers who rated quality lower may be experiencing a fit issue partly driven by poor construction rather than just sizing mismatch. The prompt explicitly requests three outputs: explanation, corrective size action, and a category-specific tip — to keep responses structured and comparable across test cases.

**Prompt Structure:**
```
A customer with the following profile ordered a clothing item:
- Height: {height} cm
- Hips: {hips} inches
- Bra size: {bra_size}, Cup size: {cup_size}
- Clothing length preference: {length}
- Size ordered: {size}
- Item category: {category}
- Quality rating given: {quality} out of 5

The ML model predicted fit as '{prediction}'.

Write a short, practical shopping guide (3-4 sentences) that:
1. Explains the fit prediction in the context of their measurements
2. Suggests what size to try instead (if the fit is not 'Fit')
3. Gives one practical tip specific to shopping for {category} items with their body profile

Keep the tone friendly and direct.
```

**Placeholders:**
- `{height}` — customer height in cm
- `{hips}` — hip measurement in inches
- `{bra_size}` — bra size number
- `{cup_size}` — cup size letter
- `{length}` — clothing length preference
- `{size}` — size ordered
- `{category}` — one of: `tops`, `bottoms`, `dresses`, `outerwear`, `wedding`, or `new` (derived from the one-hot encoded category features)
- `{quality}` — quality rating (from `quality_scaled`, reversed to original scale)
- `{prediction}` — fit label from ML model

**Example Input:**
```json
{
  "height": 170,
  "hips": 42,
  "bra_size": 36,
  "cup_size": "D",
  "length": "just right",
  "size": "38",
  "category": "dresses",
  "quality": 3,
  "prediction": "Small"
}
```

**Example Output (expected):**
> *"The large you ordered is predicted to run small for your hip and bust measurements, this dress likely pulls tightly across the hips and chest. We'd suggest trying an XL to give your proportions the room they need, particularly through the hips. When shopping for dresses with a fuller bust and hip, look for styles with an empire waist or wrap cut as they tend to be more accommodating than straight-cut fits."*

**Assumptions & Limitations:**  
The `category` value needs to be decoded from the one-hot encoded features before being inserted into the prompt (e.g., if `cat_dresses = 1`, pass `"dresses"`). The quality rating is used as soft context — the model may not always incorporate it meaningfully, which is worth noting during evaluation. As with T2, all scaled features must be inverse-transformed before use.

# 3. Implementation & API Integration Code (safe key handling)

# 4. Testing framework & Output Comparison

# 5. Analysis: Qualitative & Quantative Results

# 6. Best Prompt Selection & Justification

# 7. Integration Plan for Final System

# 8. Ethical Considerations & Limitations